In [1]:
%uv pip uninstall torchvision torchaudio
%uv pip install -U "lm_eval[hf]" accelerate transformers datasets sentencepiece peft -q

Using Python 3.12.6 environment at: /usr/local
Uninstalled 2 packages in 170ms
 - torchaudio==2.8.0+cu129
 - torchvision==0.23.0+cu129
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%uv pip install -U langdetect immutabledict -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from datetime import datetime
from huggingface_hub import login

HFTOKEN = os.environ.get("HF_TOKEN")
assert HFTOKEN, "Set HFTOKEN in Modal secrets or env"

login(token=HFTOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HFTOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HFTOKEN

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

RUN_TS = datetime.now().strftime("%Y%m%d-%H%M%S")
ROOT_DIR = f"/root/lily-eval-runs/{RUN_TS}"
CACHE_DIR = f"{ROOT_DIR}/cache"
OUT_DIR = f"{ROOT_DIR}/results"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("ROOT_DIR :", ROOT_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("OUT_DIR  :", OUT_DIR)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


ROOT_DIR : /root/lily-eval-runs/20260520-050029
CACHE_DIR: /root/lily-eval-runs/20260520-050029/cache
OUT_DIR  : /root/lily-eval-runs/20260520-050029/results


In [2]:
import torch

assert torch.cuda.is_available(), "No GPU detected"
p = torch.cuda.get_device_properties(0)

print("GPU:", p.name)
print(f"VRAM: {p.total_memory / 1e9:.1f} GB")
print("Compute capability:", f"{p.major}.{p.minor}")
print("BF16 supported:", p.major >= 8)

GPU: NVIDIA L4
VRAM: 23.7 GB
Compute capability: 8.9
BF16 supported: True


In [3]:
MODEL_NAME = "abhinav0231/Lily-1.5b-v0.3"
DEVICE = "cuda:0"

GROUP1_TASKS = [
    "hellaswag",
    "arc_challenge",
    "mmlu",
]

GROUP2_TASKS = [
    "mmlu_redux_generative",
    "gsm8k",
    "ifeval",
]

BATCH1 = 16
BATCH2 = 8

MODEL_ARGS = ",".join([
    f"pretrained={MODEL_NAME}",
    "dtype=bfloat16",
    "trust_remote_code=True",
    "attn_implementation=sdpa",
])

print("MODEL_ARGS:", MODEL_ARGS)
print("GROUP1:", GROUP1_TASKS)
print("GROUP2:", GROUP2_TASKS)

MODEL_ARGS: pretrained=abhinav0231/Lily-1.5b-v0.3,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa
GROUP1: ['hellaswag', 'arc_challenge', 'mmlu']
GROUP2: ['mmlu_redux_generative', 'gsm8k', 'gpqa', 'ifeval']


In [4]:
import os
import shlex
import subprocess

def run_lmeval(tasks, batch_size, out_name, cache_name):
    cmd = [
        "lm_eval",
        "--model", "hf",
        "--model_args", MODEL_ARGS,
        "--tasks", ",".join(tasks),
        "--batch_size", str(batch_size),
        "--apply_chat_template",
        "--device", DEVICE,
        "--use_cache", os.path.join(CACHE_DIR, cache_name),
        "--output_path", os.path.join(OUT_DIR, out_name),
    ]
    print("Running:\n" + " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True)

In [5]:
run_lmeval(
    tasks=GROUP1_TASKS,
    batch_size=BATCH1,
    out_name="group1",
    cache_name="group1.db",
)

Running:
lm_eval --model hf --model_args pretrained=abhinav0231/Lily-1.5b-v0.3,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa --tasks hellaswag,arc_challenge,mmlu --batch_size 16 --apply_chat_template --device cuda:0 --use_cache /root/lily-eval-runs/20260519-070122/cache/group1.db --output_path /root/lily-eval-runs/20260519-070122/results/group1


2026-05-19:07:01:49 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-19:07:02:11 INFO     [_cli.run:388] Selected Tasks: ['hellaswag', 'arc_challenge', 'mmlu']
2026-05-19:07:02:14 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-19:07:02:14 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'trust_remote_code': True, 'attn_implementation': 'sdpa'}
[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2026-05-19:07:02:27 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-19:07:02:30 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cud

hf ({'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 16
|                 Tasks                 |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|--------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc     |   |0.5516|±  |0.0040|
| - humanities                          |      2|none  |     0|acc     |↑  |0.4897|±  |0.0068|
|  - formal_logic                       |      1|none  |     0|acc     |↑  |0.3492|±  |0.0426|
|  - high_school_european_history       |      1|none  |     0|acc     |↑  |0.6909|±  |0.0361|
|  - high_school_us_history             |      1|none  |     0|acc     |↑  |0.6520|±  |0.0334|
|  - high_school_world_history          |      1|none  |     0|acc     |↑  |0.7257|±  |0.0290|
|  - international_law                  |      1|none  |     0|acc     |

In [5]:
run_lmeval(
    tasks=["gsm8k"],
    batch_size=8,
    out_name="gsm8k",
    cache_name="gsm8k.db",
)

Running:
lm_eval --model hf --model_args pretrained=abhinav0231/Lily-1.5b-v0.3,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa --tasks gsm8k --batch_size 8 --apply_chat_template --device cuda:0 --use_cache /root/lily-eval-runs/20260520-050029/cache/gsm8k.db --output_path /root/lily-eval-runs/20260520-050029/results/gsm8k


2026-05-20:05:01:01 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-20:05:01:16 INFO     [_cli.run:388] Selected Tasks: ['gsm8k']
2026-05-20:05:01:19 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-20:05:01:19 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'trust_remote_code': True, 'attn_implementation': 'sdpa'}
[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2026-05-20:05:01:30 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-20:05:01:32 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|█

hf ({'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.6270|±  |0.0133|
|     |       |strict-match    |     5|exact_match|↑  |0.6171|±  |0.0134|



In [6]:
run_lmeval(
    tasks=["ifeval"],
    batch_size=8,
    out_name="ifeval",
    cache_name="ifeval.db",
)

Running:
lm_eval --model hf --model_args pretrained=abhinav0231/Lily-1.5b-v0.3,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa --tasks ifeval --batch_size 8 --apply_chat_template --device cuda:0 --use_cache /root/lily-eval-runs/20260520-050029/cache/ifeval.db --output_path /root/lily-eval-runs/20260520-050029/results/ifeval


2026-05-20:05:28:57 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-20:05:29:11 INFO     [_cli.run:388] Selected Tasks: ['ifeval']
2026-05-20:05:29:14 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-20:05:29:14 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'trust_remote_code': True, 'attn_implementation': 'sdpa'}
[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2026-05-20:05:29:20 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-20:05:29:21 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|

Downloaded punkt_tab on rank 0


Generating train split: 100%|██████████████████████████████████████| 541/541 [00:00<00:00, 31296.46 examples/s]
2026-05-20:05:29:26 INFO     [evaluator_utils:446] Selected tasks:
2026-05-20:05:29:26 INFO     [evaluator_utils:480] Task: ifeval (ifeval/ifeval.yaml)
2026-05-20:05:29:26 INFO     [evaluator:314] ifeval: Using gen_kwargs: {'until': [], 'do_sample': False, 'temperature': 0.0, 'max_gen_toks': 1280}
2026-05-20:05:29:26 INFO     [api.task:312] Building contexts for ifeval on rank 0...
100%|█████████████████████████████████████████████████████████████████████| 541/541 [00:00<00:00, 13647.11it/s]
2026-05-20:05:29:26 INFO     [evaluator:585] Running generate_until requests
2026-05-20:05:29:26 INFO     [api.model:280] Loading 'generate_until' responses from cache '/root/lily-eval-runs/20260520-050029/cache/ifeval.db_rank0.db' where possible...
Checking cached requests: 100%|████████████████████████████████████████████| 541/541 [00:00<00:00, 4368.28it/s]
2026-05-20:05:29:27 INFO     

hf ({'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.5012|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.4436|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.4030|±  |0.0211|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.3438|±  |0.0204|



In [7]:
run_lmeval(
    tasks=["mmlu_redux_generative"],
    batch_size=8,
    out_name="mmlu_redux_generative",
    cache_name="mmlu_redux_generative.db",
)

Running:
lm_eval --model hf --model_args pretrained=abhinav0231/Lily-1.5b-v0.3,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa --tasks mmlu_redux_generative --batch_size 8 --apply_chat_template --device cuda:0 --use_cache /root/lily-eval-runs/20260520-050029/cache/mmlu_redux_generative.db --output_path /root/lily-eval-runs/20260520-050029/results/mmlu_redux_generative


2026-05-20:06:15:40 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-20:06:15:54 INFO     [_cli.run:388] Selected Tasks: ['mmlu_redux_generative']
2026-05-20:06:15:57 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-20:06:15:57 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'trust_remote_code': True, 'attn_implementation': 'sdpa'}
[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2026-05-20:06:16:02 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-20:06:16:03 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading

hf ({'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|                  Tasks                  |Version|Filter |n-shot|  Metric   |   |Value |   |Stderr|
|-----------------------------------------|-------|-------|-----:|-----------|---|-----:|---|-----:|
|mmlu_redux (generative)                  |      4|default|      |exact_match|   |0.6105|±  |0.0064|
| - mmlu_redux_generative::humanities     |N/A    |default|     0|exact_match|↑  |0.6647|±  |0.0129|
|  - formal_logic                         |      4|default|     0|exact_match|↑  |0.4713|±  |0.0538|
|  - high_school_european_history         |      4|default|     0|exact_match|↑  |0.7802|±  |0.0436|
|  - high_school_us_history               |      4|default|     0|exact_match|↑  |0.7500|±  |0.0435|
|  - high_school_world_history            |      4|default|     0|exact_match|↑  |0.7576|±  |0.0433|
|  - international_law   

In [8]:
run_lmeval(
    tasks=["mmlu_generative"],
    batch_size=8,
    out_name="mmlu_generative",
    cache_name="mmlu_generative.db",
)

Running:
lm_eval --model hf --model_args pretrained=abhinav0231/Lily-1.5b-v0.3,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa --tasks mmlu_generative --batch_size 8 --apply_chat_template --device cuda:0 --use_cache /root/lily-eval-runs/20260520-050029/cache/mmlu_generative.db --output_path /root/lily-eval-runs/20260520-050029/results/mmlu_generative


2026-05-20:06:34:25 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-20:06:34:42 INFO     [_cli.run:388] Selected Tasks: ['mmlu_generative']
2026-05-20:06:34:46 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-20:06:34:46 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'trust_remote_code': True, 'attn_implementation': 'sdpa'}
[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2026-05-20:06:34:54 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-20:06:34:55 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weigh

hf ({'pretrained': 'abhinav0231/Lily-1.5b-v0.3', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|                 Tasks                 |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|---------------------------------------|-------|------------|-----:|-----------|---|----:|---|-----:|
|mmlu (generative)                      |      3|get_response|      |exact_match|   |    0|±  |     0|
| - mmlu_generative::humanities         |N/A    |get_response|     0|exact_match|↑  |    0|±  |     0|
|  - formal_logic                       |      3|get_response|     0|exact_match|↑  |    0|±  |     0|
|  - high_school_european_history       |      3|get_response|     0|exact_match|↑  |    0|±  |     0|
|  - high_school_us_history             |      3|get_response|     0|exact_match|↑  |    0|±  |     0|
|  - high_school_world_history          |      3|get_response|     0|exact_match|↑  |    0|±  |     0|
|  - inte